<a href="https://colab.research.google.com/github/24nga/datafiles/blob/%EB%8D%B0%EC%9D%B4%ED%84%B0-%EB%B6%84%EC%84%9D/Colab_SBERT_BERTScore_full.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SBERT 코사인 + BERTScore(F1) 계산 및 도표 생성

입력: `sample_similarity_results_recalc.xlsx` (sheet: `all_rows`)
출력:
- `similarity_results_with_sbert_bertscore.xlsx`
- Fig4-1~4 유사도 비교 그래프(추가로 SBERT/BERTScore 버전 포함)
- `Chapter4_figures_tables_with_sbert.zip`

비교 텍스트: `gen_text` vs `best_ref_text` (이미 best match가 잡혀있는 구조)


## 1) 설치

In [1]:
!pip -q install sentence-transformers bert-score openpyxl pandas numpy scikit-learn matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 2.1 MB/s eta 0:00:00


## 2) 파일 업로드
- Colab 좌측 Files에 `sample_similarity_results_recalc.xlsx` 업로드 후 아래 경로를 맞추세요.

In [2]:
IN_PATH = "/content/sample_similarity_results_recalc.xlsx"  # 업로드한 파일명에 맞게 수정

## 3) 데이터 로드

In [3]:
import pandas as pd
import numpy as np

df_all = pd.read_excel(IN_PATH, sheet_name="all_rows")
print(df_all.shape)
df_all.head(2)


(1014, 28)


,프로젝트ID,프로젝트명,구분자,시스템명,요구사항ID,요구사항명,"핵심 요구사항 명세 (기본 동작, 입출력, 인터페이스)",제약사항,중요도,난이도,...,cosine_tfidf,best_ref_req_id,best_ref_text,dataset,핵심 요구사항 명세 (IEEE29148 문장),비고(품질점검),핵심 요구사항 명세 (원자 단위),FP 유형 근거,sbert_cosine,bertscore_f1
0,P001,차세대 여신·리스크 통합 구축,여신신청·접수,여신신청·접수,REQ_ATOMX_001,여신신청 통합 접수 및 표준화 기능,시스템은 본 시스템은 영업점,RFP 명시사항,상,중,...,0.080888,REQ_001,"여신신청 통합 접수 및 표준화 기능 본 시스템은 영업점, 인터넷뱅킹, 모바일, 기업...",chatgpt_fast,NaN,NaN,NaN,NaN,NaN,NaN
1,P001,차세대 여신·리스크 통합 구축,여신신청·접수,여신신청·접수,REQ_ATOMX_002,여신신청 통합 접수 및 표준화 기능,인터넷뱅킹,RFP 명시사항,상,중,...,0.057564,REQ_001,"여신신청 통합 접수 및 표준화 기능 본 시스템은 영업점, 인터넷뱅킹, 모바일, 기업...",chatgpt_fast,NaN,NaN,NaN,NaN,NaN,NaN


## 4) SBERT 코사인

In [4]:
from sentence_transformers import SentenceTransformer, util
import torch

cands = df_all["gen_text"].fillna("").astype(str).tolist()
refs  = df_all["best_ref_text"].fillna("").astype(str).tolist()

mask = [(len(c.strip())>0 and len(r.strip())>0) for c,r in zip(cands, refs)]
valid_idx = np.where(mask)[0]

model_name = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
model = SentenceTransformer(model_name)

cand_emb = model.encode([cands[i] for i in valid_idx], batch_size=64, convert_to_tensor=True, show_progress_bar=True)
ref_emb  = model.encode([refs[i]  for i in valid_idx], batch_size=64, convert_to_tensor=True, show_progress_bar=True)

cos = util.cos_sim(cand_emb, ref_emb).diagonal().detach().cpu().numpy()

df_all["sbert_cosine"] = np.nan
df_all.loc[valid_idx, "sbert_cosine"] = cos

df_all["sbert_cosine"].describe()


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/16 [00:00<?, ?it/s]

Batches:   0%|          | 0/16 [00:00<?, ?it/s]

,sbert_cosine
count,1014.000000
mean,0.573209
std,0.163973
min,0.000209
25%,0.485960
50%,0.594989
75%,0.687446
max,0.958632


## 5) BERTScore(F1)

In [5]:
from bert_score import score as bertscore

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

P, R, F1 = bertscore(
    cands=[cands[i] for i in valid_idx],
    refs=[refs[i] for i in valid_idx],
    model_type="xlm-roberta-large",
    lang="ko",
    device=device,
    batch_size=16,
    verbose=True,
    rescale_with_baseline=True,
)

df_all["bertscore_f1"] = np.nan
df_all.loc[valid_idx, "bertscore_f1"] = F1.detach().cpu().numpy()

df_all["bertscore_f1"].describe()


device: cpu


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/616 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

...

  0%|          | 0/65 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/64 [00:00<?, ?it/s]

done in 359.08 seconds, 2.82 sentences/sec


,bertscore_f1
count,1014.000000
mean,0.871617
std,0.027676
min,0.734938
25%,0.857714
50%,0.873080
75%,0.888600
max,0.954073


## 6) 요약 통계 + 저장

In [6]:
def summarize(df, metric_col):
    g = df.groupby("dataset")[metric_col]
    return pd.DataFrame({
        "dataset": g.count().index,
        "n": g.count().values,
        f"{metric_col}_mean": g.mean().values,
        f"{metric_col}_median": g.median().values,
        f"{metric_col}_p10": g.quantile(0.1).values,
        f"{metric_col}_p90": g.quantile(0.9).values,
    })

sum_tfidf = summarize(df_all, "cosine_tfidf")
sum_sbert = summarize(df_all, "sbert_cosine")
sum_bert  = summarize(df_all, "bertscore_f1")

summary = sum_tfidf.merge(sum_sbert, on=["dataset","n"], how="outer").merge(sum_bert, on=["dataset","n"], how="outer")
summary.sort_values("cosine_tfidf_mean", ascending=False).reset_index(drop=True).head(20)


,dataset,n,cosine_tfidf_mean,cosine_tfidf_median,cosine_tfidf_p10,cosine_tfidf_p90,sbert_cosine_mean,sbert_cosine_median,sbert_cosine_p10,sbert_cosine_p90,bertscore_f1_mean,bertscore_f1_median,bertscore_f1_p10,bertscore_f1_p90
0,chatgpt_think,166,0.286920,0.294509,0.097036,0.453068,0.661060,0.667862,0.524991,0.791061,0.889441,0.888169,0.867137,0.913826
1,chatgpt_fast,275,0.230064,0.229622,0.061646,0.402566,0.472341,0.505079,0.188682,0.708654,0.855439,0.859299,0.809701,0.897183
2,deepseek_r1_8b,45,0.175444,0.122828,0.004437,0.410261,0.692251,0.716162,0.419892,0.919934,0.882618,0.893028,0.825011,0.937979
3,gemma3_12b,164,0.163285,0.141092,0.026281,0.333854,0.619418,0.622320,0.489711,0.764170,0.882930,0.885106,0.861469,0.899542
4,gemini_think,185,0.080306,0.072853,0.025714,0.143948,0.627265,0.622771,0.514504,0.746134,0.874913,0.874077,0.859151,0.890867
5,gemini_fast,179,0.047472,0.016253,0.001400,0.133959,0.518572,0.504795,0.358891,0.695258,0.863405,0.861012,0.844355,0.888568


In [7]:
OUT_XLSX = "/content/similarity_results_with_sbert_bertscore.xlsx"
with pd.ExcelWriter(OUT_XLSX, engine="openpyxl") as w:
    summary.to_excel(w, index=False, sheet_name="summary_all")
    df_all.to_excel(w, index=False, sheet_name="all_rows")
print("saved:", OUT_XLSX)

saved: /content/similarity_results_with_sbert_bertscore.xlsx


## 7) 도표 생성
- 기존 Fig4-1~4 형식에 맞춰 SBERT/BERTScore 버전도 추가로 생성합니다.

In [8]:
import matplotlib.pyplot as plt
import os, zipfile

OUT_DIR = "/content"
os.makedirs(OUT_DIR, exist_ok=True)

def bar_with_p10p90(sum_df, metric_prefix, fig_name, ylabel, title):
    # sum_df columns: f'{metric_prefix}_mean', '_p10', '_p90'
    d = sum_df.dropna(subset=[f"{metric_prefix}_mean"]).sort_values(f"{metric_prefix}_mean", ascending=False)
    means = d[f"{metric_prefix}_mean"].values
    p10 = d[f"{metric_prefix}_p10"].values
    p90 = d[f"{metric_prefix}_p90"].values
    yerr = np.vstack([means - p10, p90 - means])

    plt.figure(figsize=(10,4))
    plt.bar(d["dataset"].values, means, yerr=yerr, capsize=4)
    plt.ylabel(ylabel)
    plt.title(title)
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    out = os.path.join(OUT_DIR, fig_name)
    plt.savefig(out, dpi=200)
    plt.close()
    return out

fig_tfidf = bar_with_p10p90(sum_tfidf, "cosine_tfidf", "Fig4-1_tfidf_cosine.png",
                            "Cosine similarity (TF-IDF char n-gram)", "Mean semantic similarity (TF-IDF cosine, p10–p90)")
fig_sbert = bar_with_p10p90(sum_sbert, "sbert_cosine", "Fig4-1_sbert_cosine.png",
                            "Cosine similarity (SBERT embedding)", "Mean semantic similarity (SBERT cosine, p10–p90)")
fig_bert = bar_with_p10p90(sum_bert, "bertscore_f1", "Fig4-1_bertscore_f1.png",
                           "BERTScore (F1)", "Mean semantic similarity (BERTScore F1, p10–p90)")

# pack
zip_path = os.path.join(OUT_DIR, "Chapter4_figures_tables_with_sbert.zip")
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for f in [OUT_XLSX, fig_tfidf, fig_sbert, fig_bert]:
        z.write(f, arcname=os.path.basename(f))
print("zip:", zip_path)

zip: /content/Chapter4_figures_tables_with_sbert.zip
